# Module 06 - Evaluation

**Duration:** 45 minutes

How do you know if your RAG system is working?
Feeling like the answers are good is not enough, especially when you change something
and want to know if it improved things.

This module covers three approaches: a manual gold dataset with Recall@k,
a simple LLM-based faithfulness check, and a brief look at RAGAS for automated scoring.

---


## 6.1 What to evaluate and why

A RAG system has two components that can fail independently.

The **retriever** can fail by not returning the right chunks.
The **generator** can fail by ignoring the context or misreading it.

This means you need to evaluate both separately, not just look at the final answer.
A system that gets the right answer for the wrong reasons will fail on harder questions.

### The evaluation triangle

Three properties we want from a RAG system:

```
              Faithful
                 ^
                 |
    Relevant ----+---- Accurate
```

- **Faithfulness** — does the answer only contain claims supported by the retrieved context?
  (Measures hallucination)
- **Relevance** — does the answer actually address what was asked?
  (Measures off-topic or evasive answers)
- **Accuracy** — is the answer factually correct?
  (Requires a ground-truth reference answer)

### Why "does it feel right?" is not enough

Human evaluation is slow, expensive, and inconsistent.
Two evaluators will often disagree on borderline cases.
More importantly, when you make a change (different chunk size, different model,
new prompt), you need to know if the change *improved* things.
You cannot measure improvement without a reproducible metric.

Even a small, imperfect automated evaluation is more useful than intuition,
because it is *consistent* and *repeatable*.

### The metrics we use in this module

- **Recall@k**: did the correct chunk appear in the top k results?
  (Measures retrieval quality; no LLM call needed)
- **Answer accuracy**: does the answer contain the expected fragment?
  (Cheap proxy for answer quality; requires a gold dataset)
- **Faithfulness**: is the answer grounded in the retrieved context?
  (Measured via an LLM judge)

We also briefly introduce **RAGAS**, a framework that automates all of this at scale.


## 6.2 Building a gold dataset

A gold dataset is a set of question-answer pairs where you know the correct answer
and, importantly, **which document or chunk contains it**.

### How to build a good gold dataset

**Manual construction (what we do here):**
- Read the documents, think of questions a real user would ask
- Record the expected answer *fragment* (a distinctive phrase from the source document)
- Record the source file — this is used for Recall@k

**LLM-assisted generation (faster for large corpora):**
- Prompt an LLM: "Given this passage, write 3 questions whose answers are in the passage"
- Manually review and filter the generated questions
- This is how RAGAS generates its test sets

### What makes a good gold question?

**Good:** "When do the paper reading sessions take place?" — specific, has a definite answer, answer is in one place  
**Bad:** "What does the AI center do?" — vague, answer spans many chunks, subjective  
**Bad:** "Is the AI center good?" — opinion question, no factual answer  

> **Small is fine for a workshop.** In production you would want 50–100 questions
> to get stable metric estimates. Below ~20 questions, the variance is too high
> to draw conclusions from small score differences (e.g. 0.83 vs 0.87 on 6 questions
> is just 0 vs 1 correct — meaningless).


In [ ]:
from ragsst.ragtool import RAGTool

tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.setup_vec_store()

# Each entry: question, expected answer fragment, source file
gold_dataset = [
    {
        'question': 'What AI workshops does the service center offer?',
        'expected_fragment': 'AI Maker Sessions',
        'source': 'aihpi-home.txt',
    },
    {
        'question': 'When do the paper reading sessions take place?',
        'expected_fragment': 'Wednesday',
        'source': 'aihpi-home.txt',
    },
    {
        'question': 'What does John McClane find in the bag he takes from a terrorist?',
        'expected_fragment': 'C-4',
        'source': 'die-hard.txt',
    },
    {
        'question': 'How does Gruber plan to escape with the bonds?',
        'expected_fragment': 'helicopter',
        'source': 'die-hard.txt',
    },
    {
        'question': 'What is Wild Tales about?',
        'expected_fragment': 'vengeance',
        'source': 'wild-tales.txt',
    },
    {
        'question': 'What does Bombita do after his car gets towed a second time?',
        'expected_fragment': 'explosives',
        'source': 'wild-tales.txt',
    },
]

print(f'Gold dataset: {len(gold_dataset)} questions')


## 6.3 Recall@k

Recall@k answers: *for a given question, does the correct source document
appear in the top k retrieved chunks?*

It measures retriever quality **independently of the generator**.
This is important: if Recall@k is low, no amount of prompt engineering will fix things.
You need to fix retrieval first.

### Interpreting Recall@k

- **Recall@1 = 1.0** — the right chunk is always the top result. Excellent.
- **Recall@3 = 1.0, Recall@1 = 0.6** — the right chunk is *in* the top 3 but not always #1.
  This means retrieval is finding it, but ranking is imprecise. Good candidate for re-ranking.
- **Recall@5 = 0.7** — 30% of the time the answer is not in the top 5 results at all.
  This is a retrieval problem: chunking, embedding model, or similarity threshold.

### What Recall@k does NOT measure

- Whether the answer is actually in the chunk (a chunk can be from the right source file
  but contain the wrong section)
- Whether the generator reads and uses the context correctly
- Whether the answer is phrased well

For those, you need the generation metrics below.


In [ ]:
def recall_at_k(dataset: list[dict], k: int = 3) -> float:
    hits = 0
    for item in dataset:
        results = tool.collection.query(
            query_texts=[item['question']],
            n_results=k,
            include=['documents', 'metadatas'],
        )
        sources = [m['source'] for m in results['metadatas'][0]]
        if item['source'] in sources:
            hits += 1
    return hits / len(dataset)


print(f"{'k':>4}  {'Recall@k':>10}")
print('-' * 18)
for k in [1, 2, 3, 5]:
    score = recall_at_k(gold_dataset, k=k)
    print(f'{k:>4}  {score:>10.2f}')


Recall@1 tells you how often the right document is the very top result.
Recall@3 tells you how often it appears somewhere in the top 3.

If Recall@3 is 1.0 but Recall@1 is 0.5, your retriever is finding the right
document but not ranking it first - that is a good candidate for re-ranking.


In [ ]:
# Per-question breakdown to see where retrieval fails
print(f"{'Question':50}  {'Hit?':6}  {'Sources returned'}")
print('-' * 90)

for item in gold_dataset:
    results = tool.collection.query(
        query_texts=[item['question']],
        n_results=3,
        include=['metadatas'],
    )
    sources = [m['source'] for m in results['metadatas'][0]]
    hit = item['source'] in sources
    print(f"{item['question'][:50]:50}  {'yes' if hit else 'NO':6}  {', '.join(set(sources))}")


## 6.4 Faithfulness check

A faithfulness check asks: *does the generated answer contain claims
that are not supported by the retrieved context?*

We will use the LLM itself as a judge ("LLM-as-judge"). This is not perfect but it is
fast and catches obvious hallucinations.

### LLM-as-judge: pros and cons

**Pros:**
- Fast and cheap (one extra LLM call per answer)
- Catches clear-cut hallucinations reliably
- Can give detailed explanations, not just pass/fail

**Cons:**
- The judge model can also hallucinate
- Inconsistent on borderline cases
- The judge can be "fooled" if the answer is plausible but wrong

### Improving the faithfulness prompt

The prompt below asks for FAITHFUL / HALLUCINATION with a brief explanation.
For production use, you can improve it by:
- Adding few-shot examples (show 2–3 examples of each verdict)
- Using a stronger judge model
- Breaking the check into sub-claims (NLI-style verification)

RAGAS (introduced in the next section) uses a more sophisticated version of this approach.


In [ ]:
import requests, json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')


def generate(prompt: str, temp: float = 0.1) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': 'llama3.2', 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


def check_faithfulness(question: str, context: str, answer: str) -> str:
    prompt = (
        'Given the context and the answer below, decide if the answer contains any '
        'information that is NOT in the context. '
        'Reply with FAITHFUL if everything in the answer is supported by the context, '
        'or HALLUCINATION followed by a brief explanation if not.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {question}\n'
        f'Answer: {answer}\n'
        'Verdict:'
    )
    return generate(prompt)


# Test on a few gold questions
for item in gold_dataset[:3]:
    context = tool.get_relevant_text(item['question'], nresults=3)
    answer = generate(tool.get_context_prompt(item['question'], context))
    verdict = check_faithfulness(item['question'], context, answer)

    print(f"Q: {item['question']}")
    print(f'A: {answer[:150]}')
    print(f'Verdict: {verdict.strip()}')
    print()


## 6.5 End-to-end scoring

We can combine retrieval and generation metrics into a simple scorecard.
Running this before *and after* any change tells you whether it helped.

### The compare-then-decide workflow

The workflow for improving a RAG system should always be:

1. **Establish a baseline** — run `evaluate_system` on the current pipeline
2. **Form a hypothesis** — "re-ranking will improve Recall@3"
3. **Make the change** — implement the improvement
4. **Measure** — run `evaluate_system` again
5. **Decide** — did the score improve? By how much? Is the added latency/cost worth it?

This is the same scientific method used in ML model development.
Never ship a change without measuring it — even "obvious" improvements sometimes hurt.


In [ ]:
def evaluate_system(dataset: list[dict], k: int = 3) -> dict:
    results = []

    for item in dataset:
        context = tool.get_relevant_text(item['question'], nresults=k)
        answer = generate(tool.get_context_prompt(item['question'], context))

        # Retrieval: did the right source appear?
        retrieval_result = tool.collection.query(
            query_texts=[item['question']], n_results=k, include=['metadatas']
        )
        sources = [m['source'] for m in retrieval_result['metadatas'][0]]
        retrieval_hit = item['source'] in sources

        # Answer: does it contain the expected fragment?
        answer_hit = item['expected_fragment'].lower() in answer.lower()

        results.append({
            'question': item['question'],
            'retrieval_hit': retrieval_hit,
            'answer_hit': answer_hit,
            'answer': answer,
        })

    recall = sum(r['retrieval_hit'] for r in results) / len(results)
    answer_acc = sum(r['answer_hit'] for r in results) / len(results)

    return {'recall_at_k': recall, 'answer_accuracy': answer_acc, 'details': results}


scores = evaluate_system(gold_dataset, k=3)

print(f"Recall@3:        {scores['recall_at_k']:.2f}")
print(f"Answer accuracy: {scores['answer_accuracy']:.2f}")
print()
for r in scores['details']:
    print(f"{'OK' if r['retrieval_hit'] else 'MISS':4} retrieval | {'OK' if r['answer_hit'] else 'MISS':4} answer | {r['question']}")


## 6.6 Introduction to RAGAS

RAGAS is an open-source framework for automated RAG evaluation.
It defines a standardised set of metrics and handles the LLM-judge calls for you.

### RAGAS metrics

| Metric | What it measures | Requires |
|--------|-----------------|---------|
| **Faithfulness** | Are all claims in the answer supported by the context? | Answer + context |
| **Answer Relevancy** | Does the answer address the question asked? | Answer + question |
| **Context Precision** | Are the retrieved chunks relevant? (Is there noise?) | Context + question |
| **Context Recall** | Did the retriever find everything needed to answer? | Context + reference answer |

### How RAGAS works

RAGAS uses an LLM (by default GPT-4, configurable) to score each metric.
For faithfulness, it:
1. Asks the LLM to decompose the answer into atomic claims
2. Asks the LLM to verify each claim against the context
3. Computes faithfulness = (supported claims) / (total claims)

This is much more robust than our simple "does the answer contain this fragment" check.

### Using RAGAS in this workshop

RAGAS requires an OpenAI API key for its default configuration.
If you don't have one, skip this section — the evaluation in 6.5 already gives you
a solid baseline.

```python
# Install RAGAS: uv pip install ragas
# Requires OPENAI_API_KEY in your environment

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

# Build a RAGAS-compatible dataset
data = {
    'question': [...],
    'answer': [...],     # generated answers
    'contexts': [...],   # list of retrieved chunks per question
    'ground_truth': [...] # reference answers from gold dataset
}

ragas_dataset = Dataset.from_dict(data)
result = evaluate(ragas_dataset, metrics=[faithfulness, answer_relevancy])
print(result)
```

Even without running it, understanding RAGAS metrics gives you a vocabulary
for discussing RAG quality with teams and stakeholders.

**Further reading**
- RAGAS paper: https://arxiv.org/abs/2309.15217
- RAGAS docs: https://docs.ragas.io/
- RAG evaluation survey: https://arxiv.org/abs/2407.01102


Now you have a baseline score for the default pipeline.
Go back to Module 05, apply one of the improvement techniques, and run this
evaluation again. Did the score improve?

---

**Exercises**

1. Add 3 more questions to `gold_dataset` from the Sherlock PDF.
   Does Recall@3 change?

2. Run `evaluate_system` with k=1, k=3, and k=5.
   At what k does adding more results stop helping?

3. Apply the re-ranking technique from Module 05 and re-run `evaluate_system`.
   Fill in the table: does Recall@3 change? Does answer accuracy change?

---

**Further reading**

- RAGAS framework: https://docs.ragas.io/
- RAG evaluation overview: https://neptune.ai/blog/evaluating-rag-pipelines
- BERGEN benchmarking library: https://arxiv.org/abs/2407.01102
